# 08 — Self-improvement loop (real data)

Drives the three learning layers using **real** RAG output (no `chunk-x` /
`chunk-y` placeholders):

1. **Chunk-quality scores** — re-rank using actual chunk IDs returned by the search index.
2. **Learned rules** — distilled from real corrections on real answers.
3. **Golden Q&A pairs** — promoted from real 👍 turns.

**Prerequisites**
- Run `06_ingestion_pipeline.ipynb` so at least one document is indexed.
- Run `07_rag_query.ipynb` once to confirm retrieval works.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.search_client import SearchService
from src.openai_client import OpenAIService
from src.cosmos_client import create_state_service
from src.rag import RAGEngine
from src.learning import LearningLoop
from src.models import FeedbackRecord, FeedbackChunkMeta

cosmos = create_state_service(); cosmos.ensure_containers()
ai = OpenAIService()
rag = RAGEngine(SearchService(), ai, cosmos)
learner = LearningLoop(cosmos, ai)

SESSION_ID = 'nb-learn-real'
USER_ID = 'nb-learn-user'
print('State backend:', type(cosmos).__name__)


INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: LocalStateService ready: ['sessions', 'documents', 'feedback', 'learned_rules', 'golden_pairs', 'chunk_quality', 'ingestion_tasks']


State backend: LocalStateService


In [ ]:
# 1. Ask real questions against the index, then submit real feedback.
#    Tune QUESTIONS to match content in YOUR indexed documents.
#    Each entry is (question, rating, correction_or_None):
#       rating='down' + correction → trains rules + penalizes cited chunks
#       rating='up'                → promotes to golden pair + boosts cited chunks
QUESTIONS = [
    ('What is the architecture of multi agent research?',  'up',   None),
    ('Summarize the key findings of the document.',        'up',   None),
    ('How many agents are described in the system?',       'down', 'Be precise with numbers — quote the exact figure from the document instead of approximating.'),
    ('What evaluation metrics are reported?',              'down', 'Always cite the page number when listing metrics.'),
    ('Describe the diagrams included in the paper.',       'down', 'Do not describe images that are merely decorative; focus on architecture diagrams only.'),
]

turns = []
for q, rating, correction in QUESTIONS:
    answer, sources, turn = rag.answer(q, session_id=SESSION_ID, user_id=USER_ID)
    chunk_ids  = [s.chunk_id for s in sources]
    chunk_meta = [FeedbackChunkMeta(chunk_id=s.chunk_id, type=s.type, page=s.page) for s in sources]

    cosmos.save_feedback(FeedbackRecord(
        session_id=SESSION_ID, turn_id=turn.id, user_id=USER_ID,
        rating=rating, correction=correction,
        question=q, answer=answer,
        chunk_ids=chunk_ids, chunk_meta=chunk_meta,
    ))
    turns.append({'q': q, 'rating': rating, 'chunk_ids': chunk_ids})
    print(f"[{rating:>4}] {q}")
    print(f"   → {len(chunk_ids)} chunks: {chunk_ids[:3]}{'…' if len(chunk_ids) > 3 else ''}")
    print(f"   → answer: {answer[:120].replace(chr(10), ' ')}…\n")

print(f'persisted {len(turns)} real feedback records ✔')


feedback seeded ✔


In [ ]:
# 2. Run all three learning layers.
stats = learner.run_once()
print('Learning stats:', stats)

print('\nRules learned:')
for r in cosmos.get_rules():
    print(' •', r.rule)

print('\nGolden pairs:')
for g in cosmos.get_golden_pairs():
    print(f'  Q: {g.question}\n     → {g.answer[:120]}…')


INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO: Learning loop done: {'feedback_count': 10, 'rules_added': 4, 'golden_added': 4, 'chunk_updates': 10}


Learning stats: {'feedback_count': 10, 'rules_added': 4, 'golden_added': 4, 'chunk_updates': 10}

Rules learned:
 • Verify the accuracy of numerical data before responding.
 • Reference specific documents or sources when providing information.
 • Ensure consistency in answers for repeated questions.
 • Cross-check information with diagrams or visual aids when available.
 • Verify the accuracy of the information before responding.

Golden pairs:
 Q: What is DocMind? → A self-improving multimodal RAG agent.
 Q: Where is the SLA defined? → On page 12 of the SRE document.
 Q: What is DocMind? → A self-improving multimodal RAG agent.


In [ ]:
# 3. Verify chunk-quality on the real chunks that were cited.
seen = set()
for t in turns:
    label = '👍' if t['rating'] == 'up' else '👎'
    for cid in t['chunk_ids']:
        if cid in seen:
            continue
        seen.add(cid)
        q = cosmos.get_chunk_quality(cid)
        if q is None:
            print(f'  {label} {cid}: (no quality record yet)')
        else:
            print(f'  {label} {cid}: good={q.times_in_good_answer}  bad={q.times_in_bad_answer}')


chunk-x: id='chunk-x' chunk_id='chunk-x' times_retrieved=0 times_in_good_answer=0 times_in_bad_answer=3 quality_score=0.0 updated_at='2026-05-05T12:01:42.789474+00:00'
chunk-y: id='chunk-y' chunk_id='chunk-y' times_retrieved=0 times_in_good_answer=2 times_in_bad_answer=0 quality_score=1.0 updated_at='2026-05-05T12:01:42.739676+00:00'


In [ ]:
# 4. Sanity check — re-ask one of the corrected questions in a fresh
#    session. The system prompt now includes the distilled rules +
#    golden pairs, so the new answer should reflect them.
followup = next(q for q, r, _ in QUESTIONS if r == 'down')
new_answer, new_sources, _ = rag.answer(followup, session_id=SESSION_ID + '-after', user_id=USER_ID)
print('Q:', followup)
print('A:', new_answer)
print('\nsources:', [(s.chunk_id, s.page, s.type) for s in new_sources])
